# nb57 - Convolutional-stem hybrid (H21): closing the last architecture family

**Error analysis / context.** Every architecture family tested on this data is flat or worse (GravNet, pairwise attention, Swin bias, relative-position bias, d=256, DRN-class pooling ideas) - except one that was never tried: convolutional locality priors on the dense 9x9 grid. PUMML, the founding ML-pileup work, was a CNN; at 10^4-10^5 events the classical result is that convolutional inductive bias matches or beats attention.

**Question.** Does a local convolutional stem (2 x 3x3 convs on the dense grid) feeding the standard transformer improve on pure token attention?

**Hypothesis.** H21: the 2026-standard hybrid (conv stem -> transformer) beats the EMA anchor. Expectation from all prior evidence: flat - this run exists to close the family with a measurement.

**Proof criterion.** EMA recipe, 2 seeds; anchor 0.0424 +/- 0.0003. Win = >0.002 overall or any E>17 bin.

In [1]:
import os, sys, copy, time, pathlib
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
REPO = pathlib.Path(os.environ['REPO_DIR']) if os.environ.get('REPO_DIR') else (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
sys.path.insert(0, str(REPO / 'scripts'))
from run_experiments import resolution, PITCH, EPS
from picocal_data import build_grid, prep
from picocal_models import QUANTILES, width_binned_calibration, CFG
OUT = REPO / 'reports' / 'predictions'
CKPT = REPO / '.scratch' / 'ckpt'; CKPT.mkdir(parents=True, exist_ok=True)
DEVICE = os.environ.get('NB57_DEVICE') or ('cuda' if torch.cuda.is_available() else 'cpu')
MODE = os.environ.get('NB57_MODE', 'full')
MBF = sorted((REPO / 'data' / 'minimum_bias').glob('*.root'))
CLF = sorted((REPO / 'data' / 'full').glob('matched_*.root'))
if MODE == 'smoke': MBF, CLF = MBF[:8], CLF[:4]
t0 = time.time()
ME = build_grid(MBF, 'minbias')
CE = build_grid(CLF, 'clean')
D = prep(4, ME, CE, ng=6)
L = 81; S = 9
N = D['X'].shape[0]; IN_DIM = D['IN_DIM']
XG = np.zeros((N, IN_DIM, S, S), np.float32)
EG = np.zeros((N, L), np.float32)
di = np.round(D['X'][:, :, 3] * D['std'][3] + D['mean'][3]).astype(int)
dj = np.round(D['X'][:, :, 4] * D['std'][4] + D['mean'][4]).astype(int)
for i in range(N):
    m = D['M'][i]
    ii = np.clip(di[i, m] + 4, 0, 8); jj = np.clip(dj[i, m] + 4, 0, 8)
    XG[i, :, ii, jj] = D['X'][i, m]
    EG[i, ii * S + jj] = D['Eraw'][i, m]
T = dict(XG=torch.from_numpy(XG).to(DEVICE), G=torch.from_numpy(D['G']).to(DEVICE),
         Y=torch.from_numpy(D['y']).unsqueeze(1).to(DEVICE), E=torch.from_numpy(EG).to(DEVICE))
ktr, kva, kte, ctr = D['ktr'], D['kva'], D['kte'], D['ctr']
y = D['y']; Et = D['Et']
QS = torch.tensor(QUANTILES, device=DEVICE)
print(f'device {DEVICE} | mode {MODE} | dense grids built {time.time()-t0:.0f}s')

minbias: 72554 events


clean: 30303 events


W=4: N 102857 (main 72554 + aux 30303), tr/va/te 50787/10883/10884, IN_DIM 16


device cuda | mode full | dense grids built 138s


In [2]:
NG = 6
class SubNetConv(nn.Module):
    def __init__(self, in_ch, la0, lb0):
        super().__init__()
        d = CFG['d']
        self.stem = nn.Sequential(nn.Conv2d(in_ch, 64, 3, padding=1), nn.GELU(),
                                  nn.Conv2d(64, d, 3, padding=1), nn.GELU())
        layer = nn.TransformerEncoderLayer(d, CFG['nhead'], dim_feedforward=4*d,
                                           dropout=CFG['dropout'], batch_first=True)
        self.enc = nn.TransformerEncoder(layer, CFG['layers'], enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(d + NG, d), nn.ReLU(), nn.Dropout(CFG['dropout']), nn.Linear(d, 3))
        self.fhead = nn.Sequential(nn.Linear(d, d // 2), nn.ReLU(), nn.Linear(d // 2, 1))
        self.la = nn.Parameter(torch.tensor(float(la0))); self.lb = nn.Parameter(torch.tensor(float(lb0)))
    def forward(self, xg, g, ecell):
        h = self.stem(xg).flatten(2).transpose(1, 2)
        h = self.enc(h)
        w = torch.sigmoid(self.fhead(h).squeeze(-1))
        base = self.la * torch.log1p((w * ecell).sum(1, keepdim=True)) + self.lb
        p = self.norm(h.mean(1))
        return base + self.head(torch.cat([p, g], 1))
def train_eval(seed, epochs, patience):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = SubNetConv(D['IN_DIM'], D['la0'], D['lb0']).to(DEVICE)
    ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(0.999))
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    tr_idx = np.concatenate([np.asarray(ktr), ctr])
    ck = CKPT / f'nb57_cnn_s{seed}.pt'
    def batches(idx, bs, sh=None):
        idx = np.asarray(idx)
        if sh is not None: idx = sh.permutation(idx)
        for j in range(0, len(idx), bs): yield torch.from_numpy(idx[j:j+bs]).to(DEVICE)
    def fwd(m_, b): return m_(T['XG'][b], T['G'][b], T['E'][b])
    def vloss(m_):
        m_.eval(); s = 0.0; k = 0
        with torch.no_grad():
            for b in batches(kva, 256):
                d = T['Y'][b] - fwd(m_, b)
                s += torch.maximum(QS * d, (QS - 1) * d).mean().item(); k += 1
        return s / max(k, 1)
    best = 1e9; bstate = None; wait = 0; ep0 = 0
    if ck.exists():
        st = torch.load(ck, map_location=DEVICE)
        model.load_state_dict(st['model']); ema.load_state_dict(st['ema'])
        opt.load_state_dict(st['opt']); sched.load_state_dict(st['sched'])
        best = st['best']; bstate = st['bstate']; wait = st['wait']; ep0 = st['ep'] + 1
        rng = np.random.default_rng(seed + 1000 * ep0)
        print(f'  resume s{seed} from ep {ep0}', flush=True)
    for ep in range(ep0, epochs):
        model.train()
        for b in batches(tr_idx, CFG['batch'], rng):
            opt.zero_grad()
            d = T['Y'][b] - fwd(model, b)
            torch.maximum(QS * d, (QS - 1) * d).mean().backward()
            opt.step()
            ema.update_parameters(model)
        sched.step()
        vv = vloss(ema.module)
        if vv < best - 1e-4: best = vv; bstate = copy.deepcopy(ema.module.state_dict()); wait = 0
        else: wait += 1
        torch.save(dict(model=model.state_dict(), ema=ema.state_dict(), opt=opt.state_dict(),
                        sched=sched.state_dict(), best=best, bstate=bstate, wait=wait, ep=ep), ck)
        if wait >= patience: break
    final = SubNetConv(D['IN_DIM'], D['la0'], D['lb0']).to(DEVICE)
    final.load_state_dict(bstate); final.eval()
    def run(idx):
        out = []
        with torch.no_grad():
            for b in batches(idx, 256): out.append(fwd(final, b).cpu().numpy())
        return np.concatenate(out)
    pe = width_binned_calibration(run(kva), run(kte), y[kva])
    return float(resolution(pe, Et[kte])['sigma_eff']), pe

In [3]:
EPOCHS = {'smoke': 2, 'full': 100}[MODE]
PATIENCE = {'smoke': 99, 'full': 15}[MODE]
SEEDS = {'smoke': [0], 'full': [0, 1]}[MODE]
TAG = '' if MODE == 'full' else '_smoke'
CSVP = OUT / f'nb57_cnn{TAG}.csv'
done = set()
if CSVP.exists():
    done = set(pd.read_csv(CSVP)['seed'])
    print('resume, done:', sorted(done))
for seed in SEEDS:
    if seed in done: print('skip', seed); continue
    t1 = time.time()
    sig, pe = train_eval(seed, EPOCHS, PATIENCE)
    np.save(OUT / f'nb57_pred{TAG}_cnn_s{seed}.npy', pe)
    row = dict(seed=seed, sigma_eff=round(sig, 4), elapsed=round(time.time()-t1))
    pd.DataFrame([row]).to_csv(CSVP, mode='a', header=not CSVP.exists() or CSVP.stat().st_size == 0, index=False)
    print(f'cnn seed {seed}: sigma_eff {sig:.4f} ({row["elapsed"]}s)', flush=True)
print(pd.read_csv(CSVP).to_string(index=False))

  resume s0 from ep 2


cnn seed 0: sigma_eff 0.0474 (927s)


cnn seed 1: sigma_eff 0.0478 (951s)


 seed  sigma_eff  elapsed
    0     0.0474      927
    1     0.0478      951


## Verdict

Anchor 0.0424 +/- 0.0003. Win = >0.002 overall or any E>17 bin; flat closes the convolutional family with a measurement.

In [4]:
te_e = Et[kte]
edges = np.quantile(te_e, np.linspace(0, 1, 7))
preds = [np.load(OUT / f'nb57_pred{TAG}_cnn_s{s}.npy') for s in SEEDS
         if (OUT / f'nb57_pred{TAG}_cnn_s{s}.npy').exists()]
if preds:
    sig = [resolution(p, te_e)['sigma_eff'] for p in preds]
    ens = np.stack(preds).mean(0)
    bins = []
    for i in range(6):
        hi = edges[i+1] + (1e-9 if i == 5 else 0)
        mm = (te_e >= edges[i]) & (te_e < hi)
        bins.append(f'{resolution(ens[mm], te_e[mm])["sigma_eff"]:.4f}')
    print(f'cnn mean {np.mean(sig):.4f} +/- {np.std(sig):.4f} | ens {resolution(ens, te_e)["sigma_eff"]:.4f}  [anchor 0.0424 +/- 0.0003]')
    print('per-bin ' + ' / '.join(bins))

cnn mean 0.0476 +/- 0.0002 | ens 0.0467  [anchor 0.0424 +/- 0.0003]
per-bin 0.0678 / 0.0534 / 0.0377 / 0.0384 / 0.0385 / 0.0433
